As starting point we take the SpatialData `.zarr` file created with Harpy, and preprocess it, so it can be visualized by Vitessce.

In [ ]:
import json
import os
import numpy as np
from spatialdata import read_zarr
from dataclasses import dataclass

from vitessce import (
    VitessceConfig,
    Component as cm,
    AnnDataWrapper,
    OmeZarrWrapper
)
from vitessce.data_utils import (
    VAR_CHUNK_SIZE,
    multiplex_img_to_ome_zarr,
    optimize_adata,
)

In [34]:
@dataclass
class Dataset:
    name: str
    description: str
    selected_gene: str | None   # gene which shows the tissue structure nicely, used as default for plotting in e.g. Vitessce

In [ ]:
def make_vitessce_dataset(name: str,
                          description: str,
                          selected_gene: str | None,
                          spatialdata_filepath: str,
                          spaceranger_output_path: str,
                          bin_sizes_um: list[int],
                          output_dir: str,
                          base_url: str) -> None:
    
    # Note about relative paths for output files:
    # When saving zarrs we'll prefix these paths so we get absolute file paths on disk,
    # and for the view config file we'll prefix them with the base_url of the web server.

    print(f"Reading {spatialdata_filepath}")
    sdata = read_zarr(spatialdata_filepath)

    rel_output_path_img = os.path.join("A", "1", "full_image.ome.zarr")
    output_path_img = os.path.join(output_dir, rel_output_path_img)
    _write_image_zarr_for_vitessce(sdata, name, output_path_img)

    for bin_size_um in bin_sizes_um:
        rel_output_path_adata = os.path.join("A", "1", f"adata_{bin_size_um:03}um.zarr")
        output_path_adata = os.path.join(output_dir, rel_output_path_adata)
        bin_side_length = _get_visium_hd_spot_diameter(spaceranger_output_path, bin_size_um)
        _write_anndata_zarr_for_vitessce(sdata,
                                         bin_size_um,
                                         bin_side_length,
                                         output_path_adata)
            
        output_path_vitessce_config = os.path.join(output_dir, f"vitessce_config_{name}_{bin_size_um:03}um.json")
        vc = _make_vitessce_config(name,
                                   description,
                                   selected_gene,
                                   base_url,
                                   rel_output_path_adata,
                                   rel_output_path_img)

        print(f'Writing {output_path_vitessce_config}')
        with open(output_path_vitessce_config, "w") as f:
            json.dump(vc.to_dict(base_url), f, indent=4)
    

def _to_square(x, y, side):
    """
    Convert an (x, y) coordinate to a polygon (square) with a given side length.
    """
    r = side / 2
    return np.array([
        [x - r, y + r],
        [x + r, y + r],
        [x + r, y - r],
        [x - r, y - r]
    ])


def _rel_path_to_url(base_url: str, rel_path: str) -> str:
    return base_url + '/'.join(rel_path.split(os.path.sep))


def _get_visium_hd_spot_diameter(spaceranger_output_path: str,
                                 bin_size_um: int) -> float:
    scalefactors_filepath = os.path.join(spaceranger_output_path, f"outs/binned_outputs/square_{bin_size_um:03}um/spatial/scalefactors_json.json")
    
    # Radius calculation is identical to https://github.com/scverse/spatialdata-io/blob/main/src/spatialdata_io/readers/visium_hd.py#L318
    with open(scalefactors_filepath) as file:
        scalefactors_bins = json.load(file)

    diameter = scalefactors_bins["spot_diameter_fullres"]  # CHECKME: in what units?   
    return diameter 


def _write_anndata_zarr_for_vitessce(sdata,
                                     bin_size_um: int,
                                     bin_side_length: float,
                                     output_path_adata: str):

    # Create AnnData and OME Zarr files for vitessce
    adata = sdata[f"square_{bin_size_um:03}um"]

    #
    num_cells = adata.obs.shape[0]
    adata.obsm["segmentations"] = np.zeros((num_cells, 4, 2))
    for i in range(num_cells):
        adata.obsm["segmentations"][i, :, :] = _to_square(
            adata.obsm["spatial"][i, 0], adata.obsm["spatial"][i, 1], bin_side_length
        )

    #
    adata = optimize_adata(
        adata,
        #obs_cols=["location_id"],   # CHECKME: some other columns? total_counts?
        obsm_keys=[ "segmentations", "spatial" ],
        optimize_X=False,   # Was: True CHECKME: what does this do? (but it convert to dense matrices in the process...)
        # Vitessce plays nicely with dense matrices saved with chunking
        # but for large matrices (whatever that means exactly) this yields to huge memory requirements
        to_dense_X=False,   # WAS: True
    )

    # Write AnnData zarr. It will be used by Vitessce as source data.
    # (Ideally we would directly use the original SpatialData object from Harpy in Vitessce, but among other potential problems,
    # Harpy uses the latest spatialdata version, which writes zarr v3, and Vitessce currently (early Dec 2025) does not support this).
    print(f'Writing {output_path_adata}')
    adata.write_zarr(output_path_adata, chunks=[adata.shape[0], VAR_CHUNK_SIZE])


def _write_image_zarr_for_vitessce(sdata,
                                   dataset_id: str,
                                   output_path_img: str):

    # Extract full resolution H&E image to separate OME Zarr store
    array = sdata[f"{dataset_id}_full_image" ][ "scale0" ][ "image" ].data.compute()
    print(f'Writing {output_path_img}')
    multiplex_img_to_ome_zarr(img_arr=array, channel_names=[ "r", "g", "b" ], axes="cyx",  output_path=output_path_img)


def _make_vitessce_config(name: str,
                          description: str,
                          selected_gene: str | None,
                          base_url: str,
                          rel_output_path_adata: str,
                          rel_output_path_img: str) -> VitessceConfig:
    
    vc = VitessceConfig(schema_version="1.0.15",
                        name=name,
                        description=description)

    dataset = vc.add_dataset(name=__name__)

    dataset.add_object(OmeZarrWrapper(img_url=_rel_path_to_url(base_url, rel_output_path_img),
                                      is_bitmask=False,
                                      name='H&E Image'))

    dataset.add_object(AnnDataWrapper(
            adata_url=_rel_path_to_url(base_url, rel_output_path_adata),
            obs_locations_path="obsm/spatial",
            obs_segmentations_path="obsm/segmentations",
            obs_feature_matrix_path="X",
            coordination_values={
                "obsType": "spot",  # tell Vitessce to consider each observation to be a "spot"
                "featureType": "gene",
                "featureValueType": "expression"
            }
        ),
    )

    spatial = vc.add_view(cm.SPATIAL, dataset=dataset)
    layer_controller = vc.add_view(cm.LAYER_CONTROLLER, dataset=dataset, props={"disableChannelsIfRgbDetected": True})
    feature_list = vc.add_view(cm.FEATURE_LIST, dataset=dataset)
    description = vc.add_view(cm.DESCRIPTION, dataset=dataset)

    spatial_segmentation_layer_dict = {"radius": 65,
                                    "stroked": True,
                                    "visible": True,
                                    "opacity": 0.5}
    
    rgb_channels_list = [{ "selection": {"c": 0},
                        "color": [255, 0, 0],
                        "visible": True,
                        "slider": [0, 255]
                      },
                      { "selection": {"c": 1},
                        "color": [0, 255, 0],
                        "visible": True,
                        "slider": [0, 255]
                      },
                      { "selection": {"c": 2},
                        "color": [0, 0, 255],
                        "visible": True,
                        "slider": [0, 255]
                      }]
    
    spatial_image_layer_list = [{"type": "raster",
                              "index": 0,
                              "opacity": 1,
                              "colormap": None,
                              "transparentColor": None,
                              "domainType": "Min/Max",
                              "channels": rgb_channels_list}]
    
    selected_features_list = [selected_gene] if selected_gene else []

    vc.link_views([spatial, layer_controller, feature_list],
                  ['obsType', 'spatialSegmentationLayer', 'spatialImageLayer', 'obsColorEncoding', 'featureSelection'],
                  ['spot', spatial_segmentation_layer_dict, spatial_image_layer_list, "geneSelection", selected_features_list])
    
    vc.layout((spatial | (description / layer_controller / feature_list)))
    return vc

In [39]:
datasets = [
    Dataset("SCA003", "SCA003: Visium HD FFPE, brain WT", "mt-Atp6"),
    # Dataset("SCA002", "SCA002: Visium HD FFPE, liver WT", None),
    # Dataset("SCA019", "SCA019: Visium HD FF, brain WT", None),
    # Dataset("SCA019", "SCA018: Visium HD FF, liver WT", None),
]

bin_sizes_um = [2, 8, 16, 32]

In [40]:
base_url = "http://localhost:3000/"

root_dir = r"F:\2024-07-UTBenchmark-SpC\data"
assert os.path.exists(root_dir)

for dataset in datasets:
    data_dir = os.path.join(root_dir, "processed", "Visium_HD", dataset.name)
    spatialdata_filepath = os.path.join(data_dir, "harpy", "sdata.zarr")
    spaceranger_output_path = os.path.join(root_dir, f"processed/Visium_HD/{dataset.name}/SpaceRanger")
    output_dir = os.path.join(data_dir, "vitessce")

    make_vitessce_dataset(dataset.name,
                          dataset.description,
                          dataset.selected_gene,
                          spatialdata_filepath,
                          spaceranger_output_path,
                          bin_sizes_um,
                          output_dir,
                          base_url)

Reading F:\2024-07-UTBenchmark-SpC\data\processed\Visium_HD\SCA003\harpy\sdata.zarr
Writing F:\2024-07-UTBenchmark-SpC\data\processed\Visium_HD\SCA003\vitessce\vitessce_config_SCA003_002um.json
Writing F:\2024-07-UTBenchmark-SpC\data\processed\Visium_HD\SCA003\vitessce\vitessce_config_SCA003_008um.json
Writing F:\2024-07-UTBenchmark-SpC\data\processed\Visium_HD\SCA003\vitessce\vitessce_config_SCA003_016um.json
Writing F:\2024-07-UTBenchmark-SpC\data\processed\Visium_HD\SCA003\vitessce\vitessce_config_SCA003_032um.json


In [ ]:
# IMPROVEME: can we hide the tissue mask by default?
# IMPROVEME: can we list the H&E image UI controls first, and below the ones for tissue?
# IMPROVEME: show a predefined gene by default (maybe one of the mitochondrial genes that are expressed almost everywhere in the tissue, like mt-Atp6?)
# IMPROVEME: can we somehow only show the bins inside the tissue?
# IMPROVEME: make the H&E image panel wider than the default half of the screen
# IMPROVEME: can we avoid that vc.export(to="files") makes copies of zarr files that we already have somewhere else? 
#            I think so, see https://python-docs.vitessce.io/notebooks/widget_brain_with_base_dir.html